# ✂️ Document Chunking Strategies

**Day 6 — Notebook 3 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Why chunking matters and what happens when you get it wrong
- Fixed-size chunking with overlap
- Recursive character text splitting (the most common production approach)
- Sentence-aware chunking
- How to choose chunk size and overlap for your use case

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" langchain-text-splitters

In [ ]:
import sys, os, re
import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
print("✅ Ready!")

---

## 1. Why Chunking Matters

The Gemini embedding model (`text-embedding-004`) has a **2,048 token limit**. But even if your text is short enough to fit, **embedding it as one big blob is often wrong**:

```
❌ Too large:   "Chapter 1... Chapter 2... Chapter 3..."  → vague average of all topics
❌ Too small:   "The"                                     → meaningless
✅ Just right:  A coherent paragraph or passage           → captures a specific idea
```

**The chunking paradox:**
- **Chunks too large** → embeddings too general, retrieval returns the right section but also lots of irrelevant text
- **Chunks too small** → embeddings lack context, poor retrieval quality
- **Chunks at sentence boundaries** → natural meaning preservation

There is no universally correct chunk size. It depends on:
- Your document type (code, prose, FAQ, legal)
- Your query style (short questions vs. long passages)
- Your LLM's context window

In [ ]:
# Sample long document for all experiments
SAMPLE_DOCUMENT = """
Introduction to Machine Learning

Machine learning is a branch of artificial intelligence that enables computers to learn from data 
without being explicitly programmed. Instead of following a fixed set of rules, machine learning 
systems identify patterns in data and use those patterns to make predictions or decisions.

There are three main types of machine learning. Supervised learning uses labelled examples to 
train a model. The model learns to map inputs to outputs based on the provided examples. Common 
applications include spam detection, image classification, and price prediction.

Unsupervised learning works with unlabelled data. The model discovers hidden structure or patterns 
without guidance. Clustering algorithms are a key example — they group similar data points together. 
Anomaly detection is another application, finding unusual patterns that deviate from the norm.

Reinforcement learning trains an agent to make decisions by rewarding good actions and penalising 
bad ones. The agent interacts with an environment and learns through trial and error. This approach 
has achieved remarkable results in game playing, robotics, and autonomous systems.

Neural networks are the foundation of modern deep learning. Inspired by the human brain, they 
consist of layers of interconnected nodes that transform inputs into outputs. Deep learning models 
with many layers can learn extremely complex representations from raw data such as images, audio, 
and text.

The training process involves optimising the model's parameters using an algorithm called gradient 
descent. The model makes predictions, compares them to the true labels using a loss function, and 
adjusts its weights to reduce the error. This process repeats over many iterations until the model 
converges to a good solution.

Overfitting occurs when a model learns the training data too well and fails to generalise to new 
examples. Techniques like dropout, regularisation, and early stopping help prevent overfitting. 
Cross-validation is used to evaluate how well a model generalises.
"""

word_count = len(SAMPLE_DOCUMENT.split())
print(f"Document word count: {word_count}")
print(f"Document character count: {len(SAMPLE_DOCUMENT)}")

---

## 2. Fixed-Size Chunking

The simplest approach: split every N characters (or words), with optional overlap.

In [ ]:
def fixed_size_chunk(text: str, chunk_size: int, overlap: int = 0) -> list[str]:
    """Split text into fixed-size character chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]


# Try different chunk sizes
for size in [200, 500, 1000]:
    chunks = fixed_size_chunk(SAMPLE_DOCUMENT, chunk_size=size, overlap=50)
    print(f"Chunk size {size} chars, overlap 50 → {len(chunks)} chunks")
    print(f"  First chunk preview: '{chunks[0][:80]}...'")
    print()

In [ ]:
# Problem: fixed-size chunks cut through sentences mid-way
chunks_200 = fixed_size_chunk(SAMPLE_DOCUMENT, chunk_size=200)
print("⚠️  Fixed-size chunks break mid-sentence:")
for i, c in enumerate(chunks_200[:3]):
    print(f"\n  Chunk {i+1}: '{c}'")  

---

## 3. Overlap: Why It Matters

Without overlap, context at chunk boundaries is split across two chunks. A query about that boundary content may not find either chunk.

In [ ]:
text = "The cat sat on the mat. The mat was red. The red colour stood out."

no_overlap = fixed_size_chunk(text, chunk_size=30, overlap=0)
with_overlap = fixed_size_chunk(text, chunk_size=30, overlap=15)

print("WITHOUT overlap:")
for i, c in enumerate(no_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print("\nWITH overlap (15 chars):")
for i, c in enumerate(with_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print("\n🔍 Notice: 'The mat was red' appears in BOTH Chunk 2 and 3 with overlap.")
print("  This ensures boundary context is captured in at least one chunk.")

---

## 4. Recursive Character Text Splitting

The most widely used production strategy. It tries to split on the most natural boundaries first (paragraphs → sentences → words → characters), only going to finer splits when necessary.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,           # Target chunk size in characters
    chunk_overlap=80,         # Overlap between consecutive chunks
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Priority order for splitting
)

chunks = splitter.split_text(SAMPLE_DOCUMENT)

print(f"Recursive splitting → {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

**Notice:** Chunks now respect paragraph and sentence boundaries — much cleaner than fixed-size.

---

## 5. Sentence-Aware Chunking

Build chunks that never break mid-sentence, targeting a word count per chunk.

In [ ]:
def sentence_chunk(text: str, target_words: int = 80, overlap_sentences: int = 1) -> list[str]:
    """
    Split text into chunks that respect sentence boundaries.
    Accumulates sentences until target_words is reached.
    Adds overlap_sentences from the previous chunk for context continuity.
    """
    # Split into sentences (simplified — handles . ! ?)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    i = 0
    
    while i < len(sentences):
        current_words = 0
        chunk_sentences = []
        
        # Add sentences until we hit the target
        while i < len(sentences) and current_words < target_words:
            chunk_sentences.append(sentences[i])
            current_words += len(sentences[i].split())
            i += 1
        
        chunks.append(" ".join(chunk_sentences))
        
        # Backtrack to add overlap from previous chunk
        if overlap_sentences > 0 and i < len(sentences):
            i -= overlap_sentences
    
    return chunks


sent_chunks = sentence_chunk(SAMPLE_DOCUMENT, target_words=60, overlap_sentences=1)

print(f"Sentence chunking → {len(sent_chunks)} chunks\n")
for i, chunk in enumerate(sent_chunks):
    word_count = len(chunk.split())
    print(f"--- Chunk {i+1} ({word_count} words) ---")
    print(chunk)
    print()

---

## 6. Code-Aware Chunking

For code documents, split on function/class boundaries rather than character count.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

python_code = """
def calculate_area(radius: float) -> float:
    """Calculate the area of a circle."""
    import math
    return math.pi * radius ** 2


def calculate_perimeter(radius: float) -> float:
    """Calculate the perimeter of a circle."""
    import math
    return 2 * math.pi * radius


class Circle:
    def __init__(self, radius: float):
        self.radius = radius
    
    def area(self) -> float:
        return calculate_area(self.radius)
    
    def perimeter(self) -> float:
        return calculate_perimeter(self.radius)
    
    def __repr__(self) -> str:
        return f"Circle(radius={self.radius})"
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=200,
    chunk_overlap=0,
)

code_chunks = python_splitter.split_text(python_code)
print(f"Code chunking → {len(code_chunks)} chunks")
for i, chunk in enumerate(code_chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

---

## 7. Choosing Chunk Size: A Framework

| Document Type | Recommended Chunk Size | Overlap | Why |
|---------------|----------------------|---------|-----|
| Dense technical docs | 400–600 chars | 10–15% | Preserve full sentences and context |
| FAQs / short answers | 100–300 chars | 0–10% | Each Q&A is already atomic |
| Long-form articles | 600–1000 chars | 10–20% | Need enough context for good embeddings |
| Code files | Per function/class | 0% | Natural code boundaries |
| Legal/financial docs | 800–1200 chars | 15–20% | Dense with cross-references |
| Conversational logs | Per turn | 1–2 turns | Dialogue is naturally segmented |

In [ ]:
# Quick chunk size analyser
def analyse_chunks(chunks: list[str], label: str) -> None:
    """Print statistics about a set of chunks."""
    sizes = [len(c) for c in chunks]
    print(f"{label}:")
    print(f"  Count:  {len(chunks)}")
    print(f"  Min:    {min(sizes)} chars")
    print(f"  Max:    {max(sizes)} chars")
    print(f"  Mean:   {sum(sizes)//len(sizes)} chars")
    print()


# Compare strategies on our sample document
analyse_chunks(fixed_size_chunk(SAMPLE_DOCUMENT, 400, 50), "Fixed-size (400, overlap 50)")

splitter_400 = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
analyse_chunks(splitter_400.split_text(SAMPLE_DOCUMENT), "Recursive (400, overlap 80)")

analyse_chunks(sentence_chunk(SAMPLE_DOCUMENT, target_words=60), "Sentence-aware (60 words)")

---

## 🏋️ Exercise 1: Chunk a Real Document

Apply chunking strategies to the multi-section document below and evaluate the quality.

In [ ]:
# Exercise 1: Multi-section chunking
multi_section_doc = """
Section 1: Installation
To install the software, download the installer from our website. Run the installer with 
administrator privileges. Follow the on-screen instructions and accept the license agreement. 
The installation typically takes 5-10 minutes. A system restart may be required after installation.

Section 2: Configuration
After installation, open the Settings panel from the main menu. Navigate to Preferences and enter 
your license key. Configure the default output directory. Set your preferred language. Save all 
changes before proceeding. The configuration file is stored at ~/.config/myapp/settings.json.

Section 3: Troubleshooting
If the application fails to start, check that your system meets the minimum requirements. 
Ensure you have at least 4GB RAM and 2GB free disk space. If the error persists, check the 
log file at ~/.logs/myapp.log. Contact support at support@example.com with the log file attached.
"""

# TODO:
# 1. Apply all three chunking strategies to this document
# 2. For the query "How do I configure the license key?",
#    which strategy produces a chunk that best answers it?
# 3. Print the top chunk from each strategy

query = "How do I configure the license key?"

# Hint: Embed the chunks and query, then find the most similar chunk
# from each strategy

---

## 🏋️ Exercise 2: Chunk Size Impact on Retrieval

Does chunk size affect retrieval quality? Test it.

In [ ]:
# Exercise 2: Chunk size vs retrieval quality
# TODO:
# 1. Take the SAMPLE_DOCUMENT from earlier
# 2. Create 3 sets of chunks with sizes: 200, 500, 1000 chars (using recursive splitter)
# 3. For each chunk set:
#    a. Embed all chunks
#    b. Search for: "How does gradient descent train a model?"
#    c. Print the top retrieved chunk
# 4. Which chunk size returns the most informative and focused answer?

test_query = "How does gradient descent train a model?"

def embed_texts(texts):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
    )
    return np.array([e.values for e in result.embeddings])


for chunk_size in [200, 500, 1000]:
    pass  # TODO: chunk, embed, search, print top result

---

## Key Takeaways

| Strategy | Best For | Watch Out For |
|----------|----------|---------------|
| **Fixed-size** | Simple, fast baseline | Breaks mid-sentence |
| **Recursive** | Most production use cases | Chunk sizes vary |
| **Sentence-aware** | Prose, articles | Uneven chunk sizes |
| **Code-aware** | Source code indexing | Language-specific setup |
| **Overlap** | All strategies | Duplicate content in index |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Chunking Strategies for RAG | Blog | [pinecone.io/learn/chunking-strategies](https://www.pinecone.io/learn/chunking-strategies/) |
| LangChain Text Splitters | Docs | [python.langchain.com/docs/modules/data_connection/document_transformers](https://python.langchain.com/docs/modules/data_connection/document_transformers/) |

---

**Next up:** [04_Document_Processing_Pipeline.ipynb](./04_Document_Processing_Pipeline.ipynb)